# Chain Domain Finder — Batch Pipeline
## Algorithm
### Per query:
**Step 1**: A → DOMIN → top-N candidates
**Step 2**: Compute pairwise score(Bi,Bj) for all pairs
**Step 3**: Pick best pair → (A, Bi, Bj)
**Step 4**: DOMO generate → predicted sequence

### Cells:
1. Setup imports
2. Load DOMIN + DOMO + FAISS (once)
3. Define `process_single_query()` function
4. Test on single query (Step-by-step)
5. Batch process all lines in gt_pred_random.jsonl

---

## Cell 1 — Setup Imports

In [2]:
import os, sys, torch, faiss, numpy as np, json, logging

sys.path.insert(0, "/sujin/PycharmProjects/DOMINO")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMO/models")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMO")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMIN")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

DEVICE = "cuda"

## Cell 2 — Load DOMIN + DOMO + FAISS (once at start)

In [3]:
FAISS_IDX = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/key_embeddings.index"
FAISS_TSV = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/sampled_proteins.tsv"

from src.DOMIN.models.ted.ted_domain_model import TedDomainModel

log.info("Loading DOMIN...")
domin = TedDomainModel(
    config_path="/sujin/PycharmProjects/DOMINO/weights/DOMIN",
    from_checkpoint="/sujin/PycharmProjects/DOMINO/weights/DOMIN/DOMIN.pt",
)
domin.to(DEVICE).eval()
TEMP = domin.model.temperature.item()
log.info(f"  DOMIN loaded (temp={TEMP:.4f})")

from src.DOMO.utils.init_utils import construct_class_by_name
log.info("Loading DOMO...")
DOMO_CONFIG = {
    "class_name": "models.Qwen3CAwDomainConditioning.Qwen3CAwDomainConditioning",
    "qwen3_type": "/sujin/Models/Qwen/Qwen3-0.6B",
    "esm_type": "/sujin/Models/esm/esm2_t33_650M_UR50D",
}
domo = construct_class_by_name(**DOMO_CONFIG, logger=logging.getLogger(__name__))
domo.load_state_dict(torch.load("/sujin/PycharmProjects/DOMINO/weights/DOMO/pytorch_model.bin", map_location=DEVICE))
domo.eval().cuda()
log.info(f"  DOMO loaded: {type(domo).__name__}")

log.info("Loading FAISS index...")
faiss_index = faiss.read_index(FAISS_IDX, faiss.IO_FLAG_MMAP)
faiss_index.nprobe = 10000
idx_to_segment = [l.strip().split("\t")[1] for l in open(FAISS_TSV)]
log.info(f"  FAISS: {faiss_index.ntotal:,} | Segments: {len(idx_to_segment):,}")

/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-05-22 14:53:59,340] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2026-05-22 14:53:59,431 INFO gcc -pthread -B /root/miniconda3/envs/DOMINO/compiler_compat -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /root/miniconda3/envs/DOMINO/include -fPIC -O2 -isystem /root/miniconda3/envs/DOMINO/include -fPIC -c /tmp/tmpdm2f964i/test.c -o /tmp/tmpdm2f964i/test.o
2026-05-22 14:53:59,451 INFO gcc -pthread -B /root/miniconda3/envs/DOMINO/compiler_compat /tmp/tmpdm2f964i/test.o -laio -o /tmp/tmpdm2f964i/a.out


 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.6
 [WARNING]  using untested triton version (3.2.0), only 1.0.0 is known to be compatible


/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @autocast_custom_fwd
/root/miniconda3/envs/DOMINO/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:66: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @autocast_custom_bwd
2026-05-22 14:54:00,558 INFO Loading DOMIN...


No lr_scheduler_kwargs provided. The default learning rate is 0.
No optimizer_kwargs provided. The default optimizer is AdamW.
Some weights of the model checkpoint were not used: ['esm.embeddings.position_ids']


2026-05-22 14:54:44,338 INFO   DOMIN loaded (temp=0.0051)
2026-05-22 14:54:44,389 INFO Loading DOMO...
2026-05-22 14:54:44,493 INFO BaseModel initialized with kwargs: {}
2026-05-22 14:54:44,493 INFO Qwen3CAwDomainConditioning initialized with gpt_type: /sujin/Models/Qwen/Qwen3-0.6B, esm_type: /sujin/Models/esm/esm2_t33_650M_UR50D
Some weights of EsmModel were not initialized from the model checkpoint at /sujin/Models/esm/esm2_t33_650M_UR50D and are newly initialized: ['contact_head.regression.bias', 'contact_head.regression.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-22 14:55:12,799 INFO   DOMO loaded: Qwen3CAwDomainConditioning
2026-05-22 14:55:12,800 INFO Loading FAISS index...
2026-05-22 14:55:42,916 INFO   FAISS: 40,782,345 | Segments: 40,782,345


## Cell 3 — Utility + `process_single_query` Function

In [18]:
def sa_to_aa(sa: str) -> str:
    """Split SA by <unk>, extract AA from each segment, rejoin with <unk>."""
    fragments = [p[::2] for p in sa.split("<unk>") if p]
    return "<unk>".join(fragments)


def process_single_query(query_sa: str, top_n: int = 20) -> dict:
    """
    Steps 1-4 for a single query SA.
    Returns result dict with domain_list and predicted_seq.
    """
    query_aa = sa_to_aa(query_sa)

    # ---- Step 1: DOMIN retrieval ----
    with torch.no_grad():
        q = domin.get_query_repr(query_sa).cpu().numpy()

    distances, indices = faiss_index.search(
        q.reshape(1, -1).astype("float32"), top_n
    )
    candidates = []
    for idx, score in zip(indices[0].tolist(), distances[0].tolist()):
        if idx == -1:
            continue
        seg = idx_to_segment[idx] if idx < len(idx_to_segment) else None
        if not seg:
            continue
        candidates.append((int(idx), float(score / TEMP), str(seg), sa_to_aa(seg)))

    candidates.sort(key=lambda x: x[1], reverse=True)

    # ---- Step 2: Pairwise compatibility via matrix multiply ----
    segs = [c[2] for c in candidates]
    with torch.no_grad():
        all_q = domin.get_query_repr(segs).cpu().numpy()  # (N, D)
        all_k = domin.get_key_repr(segs).cpu().numpy()    # (N, D)

    scores_matrix = all_q @ all_k.T / TEMP  # (N, N)

    pair_results = {}
    n = len(candidates)
    for i in range(n):
        for j in range(n):
            pair_results[(i, j)] = {
                'score_ij': float(scores_matrix[i, j]),
                'aa_i': candidates[i][3],
                'aa_j': candidates[j][3],
            }

    sorted_pairs = sorted(pair_results.items(), key=lambda x: x[1]['score_ij'], reverse=True)

    # ---- Step 3: Best pair → domain_list ----
    best_i, best_j = sorted_pairs[0][0]
    best_score = sorted_pairs[0][1]['score_ij']
    best_aa_i = sorted_pairs[0][1]['aa_i']
    best_aa_j = sorted_pairs[0][1]['aa_j']

    idx_i = candidates[best_i][0]
    idx_j = candidates[best_j][0]

    domain_list = [query_aa, best_aa_i, best_aa_j]

    # ---- Step 4: DOMO generate ----
    tok = domo.tokenizer(
        domain_list, return_tensors="pt", padding=True,
        truncation=True, max_length=512,
    )
    with torch.no_grad():
        result = domo.generate(
            domain_ids=tok.input_ids.to(DEVICE),
            domain_masks=tok.attention_mask.to(DEVICE),
            num_domains_per_protein=torch.tensor([len(domain_list)]).to(DEVICE),
        )
    predicted_seq = result["output_seqs"][0]

    return {
        'query_sa': query_sa,
        'domain_list': domain_list,
        'predicted_seq': predicted_seq,
    }

query_sa = "VaLdEpVaLaPfGqGfGfWaDaNqLlRaNgVdDtMfGhLgVwMfNdLfTdYcTnNpCqRdTdTaEpDvGsQhYhIgIgPtDpEqIkFhTkIdAfQdKqEdSkNdLfEdMlNfSwEdIkLaDlSdWlVqNpYdQaSdSqTqSaYrSaIaNlLcEpLdSdLpFqSsKlVnNgGnKcFqSySpEvFsQqRvMvKlTcLcQcVlKvDqQvAwViTkTiRkVgQkViRwNtLfVgYiTkVmKfIgNhPpAlStEa<unk>TgSiVfDgAkGtAkAmLkImQkEmDwHiIfRnApSvFqLqQvDpSdEpScGrQrSrAlMlTrAnSlAvGsAvAvFvLqDqIqVaNaFdKdFpEpEpNpYpTdSdKdNdApFsSnKvSvYrVvSvNrRtTpNdSmRdViQdSmIdGwGfIdPdFdYdPgGnInThLpQvVnWrQvQvGrIrTpNpHgLiVdAgVsDyRtThGtLdPtLpYlFvFcIqNaPc"
# result = process_single_query(query_sa, top_n=20)
# print(json.dumps(result, indent=4))

## Cell 4 — Test on Single Query (Step-by-Step)

Run this cell to test on the **first line** of gt_pred_random.jsonl.

In [8]:
jsonl_path = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/multi_domain_pipeline/gt_pred_random.jsonl"
with open(jsonl_path, "r") as f:
    first_line = json.loads(f.readline())
    print(first_line.keys())

QUERY_SA = first_line["query_domain"]
log.info(f"Query SA (len={len(QUERY_SA)}): {QUERY_SA}")
result = process_single_query(QUERY_SA, top_n=20)

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)
print(f"  Best pair score: {result['best_pair_score']:.4f}")
print(f"  domain_list ({len(result['domain_list'])} domains):")
for k, d in enumerate(result['domain_list']):
    print(f"    [{k}] len={len(d)}  {d[:50]}...")
print(f"\n  predicted_seq ({len(result['predicted_seq'])} AA):")
print(f"  {result['predicted_seq'][:80]}...")

2026-05-22 15:01:39,597 INFO Query SA (len=475): VaLdEpVaLaPfGqGfGfWaDaNqLlRaNgVdDtMfGhLgVwMfNdLfTdYcTnNpCqRdTdTaEpDvGsQhYhIgIgPtDpEqIkFhTkIdAfQdKqEdSkNdLfEdMlNfSwEdIkLaDlSdWlVqNpYdQaSdSqTqSaYrSaIaNlLcEpLdSdLpFqSsKlVnNgGnKcFqSySpEvFsQqRvMvKlTcLcQcVlKvDqQvAwViTkTiRkVgQkViRwNtLfVgYiTkVmKfIgNhPpAlStEa<unk>TgSiVfDgAkGtAkAmLkImQkEmDwHiIfRnApSvFqLqQvDpSdEpScGrQrSrAlMlTrAnSlAvGsAvAvFvLqDqIqVaNaFdKdFpEpEpNpYpTdSdKdNdApFsSnKvSvYrVvSvNrRtTpNdSmRdViQdSmIdGwGfIdPdFdYdPgGnInThLpQvVnWrQvQvGrIrTpNpHgLiVdAgVsDyRtThGtLdPtLpYlFvFcIqNaPc


dict_keys(['gt_id', 'gt_seq', 'query_domain', 'pred_domains', 'pred_seq', 'random_domains', 'random_seq'])


RuntimeError: No active exception to reraise

## Cell 5 — Batch Process All Lines

Run this cell to process **every line** in gt_pred_random.jsonl and save results.

In [ ]:
from tqdm import tqdm

jsonl_path = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/multi_domain_pipeline/gt_pred_random.jsonl"
output_path = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/multi_domain_pipeline/fast_comb_pipeline.jsonl"

log.info(f"Reading {jsonl_path}...")
with open(jsonl_path) as f:
    lines = f.readlines()
log.info(f"Total lines: {len(lines)}")

with open(output_path, "w") as w:
    for line in tqdm(lines):
        item = json.loads(line)
        query_sa = item["query_domain"]

        try:
            res = process_single_query(query_sa, top_n=20)
            w.write(f"{json.dumps(res)}\n")
            w.flush()

        except Exception as e:
            log.error(f"  → Failed: {e}")

    log.info("Done. All results saved.")

2026-05-22 15:40:33,644 INFO Reading /sujin/Datasets/TED/embedding/afdb_cluster_power0.75/multi_domain_pipeline/gt_pred_random.jsonl...
2026-05-22 15:40:33,740 INFO Total lines: 704
 15%|█████████████████████████████████▋                                                                                                                                                                                                    | 103/704 [45:52<3:35:26, 21.51s/it]